# Experiment 7e — Pure-DL: 7a-tabular MLP + learnable covariate gates (L1-penalized)

Direct recovery from 7d's failure. Diagnosis from 7d:

1. **Gates didn't train** — sigmoid scalars feeding into a Linear layer have no incentive to leave their 0.5 init because the next layer can absorb any constant scaling. Std across 9 runs was under 0.012.
2. **Past encoder didn't replace the engineered features** — the 7a-tabular MLP's 32 lag/lead/rolling features were doing real work; the encoder context vector didn't subsume them. Hybrid solo: 16.02; 7a-tabular MLP solo: 14.52.

Both findings have specific architectural fixes. This notebook applies both:

- **Keep the 7a-tabular MLP exactly** — same 52 features (11 covariates + 7 time + 1 series_mean + 32 engineered + 1 sid embed), same `[256, 256, 128]` head with BatchNorm/ReLU/Dropout, same per-series target normalization, same 3-fold × inner-val protocol.
- **Add one gate per *covariate***, not per feature. Each covariate's gate multiplies the raw value AND every engineered feature derived from it (e.g., the gate on `queue_pressure_forecast` also gates `queue_pressure_forecast_lag24`, `_roll6m`, etc.). So `gate = 0` genuinely means *"this covariate is fully dropped from the model"*, not the looser "its raw value is zeroed but its lag features still feed it back in."
- **L1 penalty on the gate activations**: `loss += λ · Σ sigmoid(gates_logits)`. This is the load-bearing change from 7d — without it, the gates have no reason to move. `λ=0.01` initially (puts the penalty at ~15% of MAE loss at convergence); easily tunable in §0.

Decision shape, same as 7a/7d:

- If solo CV-WAPE ≤ **14.52** (matches 7a-tabular MLP) → success: the L1-penalized gates didn't cost accuracy and gave us the professor-required feature-selection answer. Ship as the pure-DL submission.
- If 14.52 < solo CV-WAPE ≤ 14.78 (better than LGBM but worse than 7a) → the L1 penalty was too aggressive; some real-signal gates got squashed. Drop `λ` to 0.005 or 0.002 and re-run.
- If > 14.78 → unexpected, investigate. Likely `λ` is way too high and even strong features got gated to 0.

---
## §0. Configuration

Identical to 7a-tabular §0 except the gate-specific knobs at the bottom.

In [1]:
from pathlib import Path

DATA_DIR     = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR   = Path("/kaggle/working")
# DATA_DIR   = Path("data")
# OUTPUT_DIR = Path("experiment-7")

TRAIN_PATH     = DATA_DIR / "train.csv"
VAL_INPUT_PATH = DATA_DIR / "validation_input.csv"
VAL_INDEX_PATH = DATA_DIR / "forecast_index_validation.csv"

SEED         = 42
N_FOLDS      = 3
EVAL_HORIZON = 336

DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
TIME_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos",
             "doy_sin", "doy_cos", "is_weekend"]
TOP_DOM   = ["queue_pressure_forecast", "network_pressure_forecast",
             "workload_intensity", "shock_risk", "event_load_forecast"]
TOP3_ROLL = TOP_DOM[:3]
LAGS_DOM, LEADS_DOM, ROLL_WINS = [1, 24], [1, 24], [6, 24]

BASELINE_LGBM = dict(
    n_estimators=600, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
    reg_lambda=1.0, verbose=-1,
)

# ── MLP config — identical to 7a-tabular ─────────────────────────────────────
MLP_HIDDEN          = [256, 256, 128]
MLP_DROPOUT         = 0.2
MLP_EMBED_DIM       = 16
MLP_BATCH           = 1024
MLP_LR              = 1e-3
MLP_WEIGHT_DECAY    = 1e-4
MLP_EPOCHS          = 30
MLP_EARLY_STOP      = 5
MLP_SEEDS           = [42, 101, 202]

# ── Gate config — new for this experiment ────────────────────────────────────
GATE_L1_LAMBDA      = 0.01   # L1 weight on Σ sigmoid(gates). 0 = no penalty (replicates 7d's pathology).

# Decision thresholds (carry over from 7a)
BEATS_MLP_THRESHOLD  = 14.52
BEATS_LGBM_THRESHOLD = 14.78
R_STRONG_DIVERSITY   = 0.85

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR    = {DATA_DIR}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"MLP arch    : {MLP_HIDDEN}, dropout {MLP_DROPOUT}, embed_dim {MLP_EMBED_DIM}")
print(f"Train       : batch {MLP_BATCH}, lr {MLP_LR}, epochs {MLP_EPOCHS} "
      f"(early stop {MLP_EARLY_STOP}), seeds {MLP_SEEDS}")
print(f"Gate L1 λ   : {GATE_L1_LAMBDA}")
print(f"Decision    : beat 7a-MLP @ {BEATS_MLP_THRESHOLD}, beat LGBM @ {BEATS_LGBM_THRESHOLD}")

DATA_DIR    = /kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data
OUTPUT_DIR  = /kaggle/working
MLP arch    : [256, 256, 128], dropout 0.2, embed_dim 16
Train       : batch 1024, lr 0.001, epochs 30 (early stop 5), seeds [42, 101, 202]
Gate L1 λ   : 0.01
Decision    : beat 7a-MLP @ 14.52, beat LGBM @ 14.78


---
## §1. Setup & data loading — verbatim 7a-tabular §1

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

def wape(y_true, y_pred, eps=1e-8):
    return np.sum(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / \
           (np.sum(np.abs(y_true)) + eps) * 100

train_df  = pd.read_csv(TRAIN_PATH,     parse_dates=["timestamp"])
val_input = pd.read_csv(VAL_INPUT_PATH, parse_dates=["timestamp"])
val_index = pd.read_csv(VAL_INDEX_PATH, parse_dates=["timestamp"])

sid_to_code = {sid: i for i, sid in enumerate(sorted(train_df["series_id"].unique()))}
N_SERIES = len(sid_to_code)

print(f"train_df: {train_df.shape}, val_input: {val_input.shape}, # series: {N_SERIES}")

PyTorch device: cuda
  GPU: Tesla T4
train_df: (414720, 25), val_input: (32256, 24), # series: 96


---
## §2. Feature engineering — verbatim 7a-tabular §2, plus the gate-index map

After building the 52-feature vector, we compute `gate_idx_map[i]` for each numeric feature: the index (0–10) of the covariate this feature derives from, or `-1` if the feature isn't covariate-derived (time encodings, `series_mean`). The gate module uses this map to expand 11 learned gates into a length-52 multiplier vector.

In [3]:
def impute_domain_features(df, cols):
    df = df.sort_values(["series_id", "timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(
        lambda x: x.ffill().bfill()
    )
    return df

def add_time_features(df, time_col="timestamp"):
    df = df.copy()
    t = pd.to_datetime(df[time_col])
    df["hour_sin"]   = np.sin(2 * np.pi * t.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * t.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    df["doy_sin"]    = np.sin(2 * np.pi * t.dt.dayofyear / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * t.dt.dayofyear / 365)
    df["is_weekend"] = (t.dt.dayofweek >= 5).astype(float)
    return df

def add_temporal_features(df, lag_cols, roll_cols, lags, leads, windows):
    df = df.copy().sort_values(["series_id", "timestamp"])
    new = []
    for col in lag_cols:
        g = df.groupby("series_id", sort=False)[col]
        for L in lags:
            c = f"{col}_lag{L}";  df[c] = g.shift(L);  new.append(c)
        for L in leads:
            c = f"{col}_lead{L}"; df[c] = g.shift(-L); new.append(c)
    for col in roll_cols:
        g = df.groupby("series_id", sort=False)[col]
        for W in windows:
            c_m = f"{col}_roll{W}m"
            df[c_m] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).mean())
            new.append(c_m)
            c_s = f"{col}_roll{W}s"
            df[c_s] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).std().fillna(0))
            new.append(c_s)
    return df, new

base = add_time_features(impute_domain_features(train_df, DOMAIN_FEAT_COLS))
base["sid_code"] = base["series_id"].map(sid_to_code).astype("int32")
base, TEMPORAL_FEATS = add_temporal_features(
    base, lag_cols=TOP_DOM, roll_cols=TOP3_ROLL,
    lags=LAGS_DOM, leads=LEADS_DOM, windows=ROLL_WINS,
)

NUMERIC_FEATS  = DOMAIN_FEAT_COLS + TIME_COLS + ["series_mean"] + TEMPORAL_FEATS
FEATS_WITH_SID = NUMERIC_FEATS + ["sid_code"]

# Gate-index map: for each numeric feature, which covariate (0..10) does it derive from? -1 if none.
def build_gate_idx_map(numeric_feats, covariate_cols):
    out = []
    for f in numeric_feats:
        if f in covariate_cols:
            out.append(covariate_cols.index(f))
        else:
            matched = -1
            for ci, c in enumerate(covariate_cols):
                if f.startswith(c + "_"):
                    matched = ci
                    break
            out.append(matched)
    return out

GATE_IDX_MAP = build_gate_idx_map(NUMERIC_FEATS, DOMAIN_FEAT_COLS)

# Sanity check the map
n_gated = sum(1 for i in GATE_IDX_MAP if i >= 0)
n_not   = sum(1 for i in GATE_IDX_MAP if i <  0)
print(f"Numeric features: {len(NUMERIC_FEATS)}  ({n_gated} gated  /  {n_not} not-gated)")
print("\nGate assignment per covariate:")
for ci, cov in enumerate(DOMAIN_FEAT_COLS):
    feats = [NUMERIC_FEATS[i] for i, gi in enumerate(GATE_IDX_MAP) if gi == ci]
    print(f"  [{ci:>2}] {cov:<40}  ({len(feats)} features)")
print("\nNot-gated features:", [NUMERIC_FEATS[i] for i, gi in enumerate(GATE_IDX_MAP) if gi < 0])

Numeric features: 51  (43 gated  /  8 not-gated)

Gate assignment per covariate:
  [ 0] workload_intensity                        (9 features)
  [ 1] demand_forecast                           (1 features)
  [ 2] upstream_quality_forecast                 (1 features)
  [ 3] promotion_intensity                       (1 features)
  [ 4] shock_risk                                (5 features)
  [ 5] maintenance_known                         (1 features)
  [ 6] unit_reliability_forecast                 (1 features)
  [ 7] queue_pressure_forecast                   (9 features)
  [ 8] network_pressure_forecast                 (9 features)
  [ 9] event_load_forecast                       (5 features)
  [10] service_irregularity_risk_forecast        (1 features)

Not-gated features: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'is_weekend', 'series_mean']


---
## §3. CV framework — verbatim 7a-tabular §3

In [4]:
def fold_split(end_off, length=EVAL_HORIZON):
    tr_list, te_list = [], []
    for _, g in base.groupby("series_id", sort=False):
        g = g.sort_values("timestamp")
        cut = len(g) - end_off
        tr_list.append(g.iloc[:cut])
        te_list.append(g.iloc[cut:cut + length])
    tr = pd.concat(tr_list, ignore_index=True)
    te = pd.concat(te_list, ignore_index=True)
    sm = tr.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
    return tr.merge(sm, on="series_id"), te.merge(sm, on="series_id")

def fold_offset(k):
    return (N_FOLDS - k) * EVAL_HORIZON

def fit_lgbm_predict(tr, te, feats, params, objective, categorical=None, seed=SEED):
    model = lgb.LGBMRegressor(**params, objective=objective, random_state=seed)
    fit_kw = {} if not categorical else {"categorical_feature": categorical}
    model.fit(tr[feats], tr["target"].to_numpy(float), **fit_kw)
    return model.predict(te[feats]), model

print("CV framework ready.")

CV framework ready.


---
## §4. LightGBM-MAE anchor — must land at 14.78 ± 0.5

In [5]:
print("Running LightGBM-MAE baseline...")
oof_lgbm_rows, per_fold_lgbm = [], []
for k in range(N_FOLDS):
    tr, te = fold_split(fold_offset(k))
    preds, _ = fit_lgbm_predict(tr, te, FEATS_WITH_SID, BASELINE_LGBM,
                                "regression_l1", categorical=["sid_code"])
    per_fold_lgbm.append(wape(te["target"], preds))
    oof_lgbm_rows.append(te.assign(pred_lgbm=preds, fold=k)[
        ["series_id", "timestamp", "target", "pred_lgbm", "fold"]
    ])
oof_lgbm = pd.concat(oof_lgbm_rows, ignore_index=True)
lgbm_arr = np.array(per_fold_lgbm)

print(f"\n{'fold':>6}  {'WAPE':>8}")
for k, w in enumerate(lgbm_arr): print(f"  {k:>4}    {w:>7.3f}")
print(f"  ────")
print(f"  mean    {lgbm_arr.mean():>7.3f}")
print(f"  std     {lgbm_arr.std():>7.3f}")

LGBM_WAPE = lgbm_arr.mean()
assert abs(LGBM_WAPE - 14.78) < 0.5, f"LGBM anchor drifted: {LGBM_WAPE:.3f}"
print(f"\n✅ LGBM anchor reproduces (within 0.5 of 14.78).")

Running LightGBM-MAE baseline...

  fold      WAPE
     0     14.082
     1     15.581
     2     14.683
  ────
  mean     14.782
  std       0.616

✅ LGBM anchor reproduces (within 0.5 of 14.78).


---
## §5. `CovariateGate` + `GatedSeriesMLP` + training utilities

Three things, two of them verbatim from 7a-tabular §5:

- **`CovariateGate`** (new) — holds 11 logits, one per covariate. In forward, expands them via `GATE_IDX_MAP` into a length-52 multiplier vector and elementwise-multiplies the numeric input. Features with `gate_idx_map[i] == -1` (time / series_mean) get a multiplier of 1.0 (no gating).
- **`GatedSeriesMLP`** (modified from 7a's `SeriesMLP`) — same architecture except the `CovariateGate` is the first thing applied to the numeric input.
- **`prepare_fold_tensors`** and **`train_mlp_one_seed`** (verbatim from 7a-tabular). The only training-loop change is adding `λ · Σ sigmoid(model.gate.gates_logits)` to the MAE loss.

In [6]:
class CovariateGate(nn.Module):
    "One learnable logit per covariate, expanded to all derived features via idx_map."
    def __init__(self, n_covariates, gate_idx_map):
        super().__init__()
        self.gates_logits = nn.Parameter(torch.zeros(n_covariates))
        self.register_buffer(
            "idx_map",
            torch.tensor(gate_idx_map, dtype=torch.long)
        )
    def forward(self, x):
        # x: (B, n_features). idx_map: (n_features,) with -1 for ungated.
        gates_sig = torch.sigmoid(self.gates_logits)
        # Build per-feature multiplier: 1.0 where idx_map < 0, else gates_sig[idx_map]
        idx_clamped = self.idx_map.clamp(min=0)
        per_feat = gates_sig[idx_clamped]
        per_feat = torch.where(self.idx_map < 0,
                               torch.ones_like(per_feat),
                               per_feat)
        return x * per_feat.unsqueeze(0)
    def values(self):
        with torch.no_grad():
            return torch.sigmoid(self.gates_logits).cpu().numpy()


class GatedSeriesMLP(nn.Module):
    "7a-tabular's SeriesMLP, with a CovariateGate applied to the numeric input."
    def __init__(self, n_numeric, n_series, embed_dim, hidden_dims, dropout,
                 n_covariates, gate_idx_map):
        super().__init__()
        self.gate = CovariateGate(n_covariates, gate_idx_map)
        self.series_emb = nn.Embedding(n_series, embed_dim)
        in_dim = n_numeric + embed_dim
        layers = []
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h),
                       nn.ReLU(inplace=True), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x_num, sid):
        gated = self.gate(x_num)
        e = self.series_emb(sid)
        x = torch.cat([gated, e], dim=1)
        return self.net(x).squeeze(-1)


def prepare_fold_tensors(tr_df, te_df, numeric_feats, sid_col="sid_code", target_col="target"):
    feat_scaler = RobustScaler().fit(tr_df[numeric_feats].astype(float))
    tr_X = np.nan_to_num(feat_scaler.transform(tr_df[numeric_feats].astype(float)), nan=0.0)
    te_X = np.nan_to_num(feat_scaler.transform(te_df[numeric_feats].astype(float)), nan=0.0)

    stats = tr_df.groupby(sid_col)[target_col].agg(["mean", "std"]).reset_index()
    stats["std"] = stats["std"].fillna(1.0).clip(lower=1e-3)
    mu = stats.set_index(sid_col)["mean"]
    sd = stats.set_index(sid_col)["std"]
    tr_mu = tr_df[sid_col].map(mu).to_numpy()
    tr_sd = tr_df[sid_col].map(sd).to_numpy()
    te_mu = te_df[sid_col].map(mu).to_numpy()
    te_sd = te_df[sid_col].map(sd).to_numpy()
    tr_y_norm = (tr_df[target_col].to_numpy(float) - tr_mu) / tr_sd

    return dict(
        tr_X=torch.tensor(tr_X, dtype=torch.float32),
        te_X=torch.tensor(te_X, dtype=torch.float32),
        tr_sid=torch.tensor(tr_df[sid_col].to_numpy(np.int64)),
        te_sid=torch.tensor(te_df[sid_col].to_numpy(np.int64)),
        tr_y_norm=torch.tensor(tr_y_norm, dtype=torch.float32),
        tr_y_raw=tr_df[target_col].to_numpy(float),
        te_y_raw=te_df[target_col].to_numpy(float),
        te_mu=te_mu, te_sd=te_sd,
    )


def train_gated_mlp_one_seed(fold_tensors, n_numeric, seed, l1_lambda):
    "Train one seed; return (test_preds_unnormalized, gate_values, best_val_l1, last_l1_penalty)."
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = GatedSeriesMLP(
        n_numeric=n_numeric, n_series=N_SERIES,
        embed_dim=MLP_EMBED_DIM, hidden_dims=MLP_HIDDEN, dropout=MLP_DROPOUT,
        n_covariates=len(DOMAIN_FEAT_COLS), gate_idx_map=GATE_IDX_MAP,
    ).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MLP_EPOCHS)
    loss_fn = nn.L1Loss()

    n_tr = fold_tensors["tr_X"].shape[0]
    perm = torch.randperm(n_tr, generator=torch.Generator().manual_seed(seed))
    cut = int(0.9 * n_tr)
    tr_idx, val_idx = perm[:cut], perm[cut:]

    tr_loader = DataLoader(
        TensorDataset(
            fold_tensors["tr_X"][tr_idx],
            fold_tensors["tr_sid"][tr_idx],
            fold_tensors["tr_y_norm"][tr_idx],
        ),
        batch_size=MLP_BATCH, shuffle=True, drop_last=False,
    )
    val_X = fold_tensors["tr_X"][val_idx].to(DEVICE)
    val_sid = fold_tensors["tr_sid"][val_idx].to(DEVICE)
    val_y_norm = fold_tensors["tr_y_norm"][val_idx].numpy()

    best_val, patience, best_state = float("inf"), 0, None
    last_l1 = float("nan")

    for epoch in range(MLP_EPOCHS):
        model.train()
        for x, sid, y in tr_loader:
            x, sid, y = x.to(DEVICE), sid.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            pred = model(x, sid)
            mae = loss_fn(pred, y)
            l1  = torch.sum(torch.sigmoid(model.gate.gates_logits))
            loss = mae + l1_lambda * l1
            loss.backward()
            opt.step()
        sched.step()
        last_l1 = float(l1.detach())

        model.eval()
        with torch.no_grad():
            val_pred = model(val_X, val_sid).cpu().numpy()
        val_mae = float(np.mean(np.abs(val_pred - val_y_norm)))

        if val_mae < best_val - 1e-5:
            best_val   = val_mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience   = 0
        else:
            patience += 1
            if patience >= MLP_EARLY_STOP:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        te_X = fold_tensors["te_X"].to(DEVICE)
        te_sid = fold_tensors["te_sid"].to(DEVICE)
        te_pred_norm = model(te_X, te_sid).cpu().numpy()
    te_pred = te_pred_norm * fold_tensors["te_sd"] + fold_tensors["te_mu"]
    te_pred = np.clip(te_pred, 0, None)
    return te_pred, model.gate.values(), best_val, last_l1


print("GatedSeriesMLP + training utilities ready.")

GatedSeriesMLP + training utilities ready.


---
## §6. CV run — 3 folds × 3 seeds, with L1-penalized gates

In [7]:
import time

oof_mlp_rows, per_fold_mlp, per_fold_seed_spread = [], [], []
gate_vals_per_run = []   # (fold, seed, np.array(11))

for k in range(N_FOLDS):
    fold_t0 = time.time()
    tr, te = fold_split(fold_offset(k))
    tensors = prepare_fold_tensors(tr, te, NUMERIC_FEATS)

    seed_preds, seed_wapes = [], []
    for s in MLP_SEEDS:
        seed_t0 = time.time()
        p, gates, val_loss, l1_final = train_gated_mlp_one_seed(
            tensors, n_numeric=len(NUMERIC_FEATS),
            seed=s, l1_lambda=GATE_L1_LAMBDA,
        )
        seed_preds.append(p)
        gate_vals_per_run.append((k, s, gates))
        sw = wape(tensors["te_y_raw"], p)
        seed_wapes.append(sw)
        print(f"  fold {k} seed {s:>4}: WAPE = {sw:.3f}, val_L1 = {val_loss:.4f}, "
              f"gate_L1_sum = {l1_final:.3f}, time = {time.time() - seed_t0:.1f}s")

    avg_pred = np.mean(np.stack(seed_preds, axis=0), axis=0)
    fold_w = wape(tensors["te_y_raw"], avg_pred)
    spread = max(seed_wapes) - min(seed_wapes)
    per_fold_mlp.append(fold_w)
    per_fold_seed_spread.append(spread)
    print(f"  ── fold {k}: avg WAPE = {fold_w:.3f}, seed spread = {spread:.3f}, "
          f"time = {time.time() - fold_t0:.1f}s\n")

    oof_mlp_rows.append(te.assign(pred_mlp_gated=avg_pred, fold=k)[
        ["series_id", "timestamp", "target", "pred_mlp_gated", "fold"]
    ])

oof_mlp = pd.concat(oof_mlp_rows, ignore_index=True)
mlp_arr = np.array(per_fold_mlp)
MLP_WAPE = mlp_arr.mean()

print("=== Gated-MLP CV summary ===")
print(f"{'fold':>6}  {'WAPE':>8}  {'seed spread':>12}")
for k, (w, sp) in enumerate(zip(mlp_arr, per_fold_seed_spread)):
    print(f"  {k:>4}    {w:>7.3f}    {sp:>10.3f}")
print(f"  ────")
print(f"  mean    {MLP_WAPE:>7.3f}")
print(f"  std     {mlp_arr.std():>7.3f}")

print(f"\nAnchors  — LGBM (§4): {LGBM_WAPE:.3f}   |  MLP (7a, no gates): {BEATS_MLP_THRESHOLD:.2f}")
print(f"Gated MLP (§6)        : {MLP_WAPE:.3f}")
print(f"Δ vs LGBM             : {MLP_WAPE - LGBM_WAPE:+.3f}")
print(f"Δ vs 7a-MLP (no gates): {MLP_WAPE - BEATS_MLP_THRESHOLD:+.3f}")

  fold 0 seed   42: WAPE = 14.075, val_L1 = 0.2796, gate_L1_sum = 1.198, time = 107.9s
  fold 0 seed  101: WAPE = 14.087, val_L1 = 0.2767, gate_L1_sum = 1.208, time = 103.1s
  fold 0 seed  202: WAPE = 13.907, val_L1 = 0.2753, gate_L1_sum = 1.203, time = 102.9s
  ── fold 0: avg WAPE = 13.931, seed spread = 0.180, time = 315.4s

  fold 1 seed   42: WAPE = 15.525, val_L1 = 0.2736, gate_L1_sum = 1.090, time = 106.9s
  fold 1 seed  101: WAPE = 15.419, val_L1 = 0.2757, gate_L1_sum = 1.089, time = 106.8s
  fold 1 seed  202: WAPE = 15.527, val_L1 = 0.2734, gate_L1_sum = 1.094, time = 106.6s
  ── fold 1: avg WAPE = 15.372, seed spread = 0.108, time = 322.1s

  fold 2 seed   42: WAPE = 14.563, val_L1 = 0.2717, gate_L1_sum = 1.004, time = 110.7s
  fold 2 seed  101: WAPE = 14.544, val_L1 = 0.2764, gate_L1_sum = 1.010, time = 110.1s
  fold 2 seed  202: WAPE = 14.567, val_L1 = 0.2708, gate_L1_sum = 1.004, time = 110.5s
  ── fold 2: avg WAPE = 14.477, seed spread = 0.023, time = 333.2s

=== Gated-MLP

---
## §7. Headline — learned gate values + residual decorrelation

The two deliverables. Gate values are sorted descending; std across (fold × seed) runs tells you how stable the verdict is — a low-std DROP is genuinely a feature the model agrees is useless, a high-std PARTIAL means seeds disagree and the verdict isn't trustworthy.

In [8]:
gate_matrix = np.stack([g for (_, _, g) in gate_vals_per_run], axis=0)
gate_mean = gate_matrix.mean(axis=0)
gate_std  = gate_matrix.std(axis=0)
order     = np.argsort(-gate_mean)

print("=== Learned covariate gates (sigmoid scalar per covariate) ===")
print(f"  L1 λ = {GATE_L1_LAMBDA}.  Averaged over {gate_matrix.shape[0]} (fold × seed) runs.\n")
print(f"  {'covariate':<40}  {'mean':>6}  {'std':>6}  {'verdict':>10}")
print("  " + "-" * 70)
for i in order:
    v_mean, v_std = gate_mean[i], gate_std[i]
    if   v_mean > 0.60: verdict = "KEEP"
    elif v_mean > 0.20: verdict = "PARTIAL"
    else:               verdict = "DROP"
    print(f"  {DOMAIN_FEAT_COLS[i]:<40}  {v_mean:>6.3f}  {v_std:>6.3f}  {verdict:>10}")

# Compare against the data-analysis verdicts to see if the model agrees
DA_VERDICT = {
    "queue_pressure_forecast":             "STRONG",
    "network_pressure_forecast":           "STRONG",
    "workload_intensity":                  "STRONG",
    "shock_risk":                          "STRONG",
    "event_load_forecast":                 "STRONG",
    "demand_forecast":                     "MODERATE",
    "maintenance_known":                   "MODERATE",
    "upstream_quality_forecast":           "MODERATE",
    "promotion_intensity":                 "WEAK",
    "unit_reliability_forecast":           "WEAK",
    "service_irregularity_risk_forecast":  "WEAK",
}
print("\n  vs. exp-data-analysis pooled-R² verdicts:")
for i in order:
    print(f"    {DOMAIN_FEAT_COLS[i]:<40}  gate={gate_mean[i]:.3f}  DA={DA_VERDICT[DOMAIN_FEAT_COLS[i]]}")

# Residual correlation diagnostic
joined = oof_lgbm.merge(
    oof_mlp[["series_id", "timestamp", "pred_mlp_gated", "fold"]],
    on=["series_id", "timestamp", "fold"], how="inner",
)
assert len(joined) == len(oof_lgbm) == len(oof_mlp), "OOF row counts diverged."
joined["resid_lgbm"] = joined["target"] - joined["pred_lgbm"]
joined["resid_mlp"]  = joined["target"] - joined["pred_mlp_gated"]
r_pooled = float(np.corrcoef(joined["resid_lgbm"], joined["resid_mlp"])[0, 1])
r_per_fold = [float(np.corrcoef(
    joined.loc[joined.fold == k, "resid_lgbm"],
    joined.loc[joined.fold == k, "resid_mlp"],
)[0, 1]) for k in range(N_FOLDS)]

print("\n=== Pearson r(residual_LGBM, residual_gatedMLP) ===")
for k, r in enumerate(r_per_fold):
    print(f"  fold {k}: r = {r:.4f}")
print(f"  pooled   r = {r_pooled:.4f}")

print("\n=== Verdict ===")
if MLP_WAPE <= BEATS_MLP_THRESHOLD:
    print(f"✅ Gated MLP CV-WAPE {MLP_WAPE:.3f} ≤ {BEATS_MLP_THRESHOLD:.2f} (7a-MLP). "
          f"L1-penalized gates did not cost accuracy — ship as the pure-DL submission. "
          f"Build 7f (refit on full train, write validation_predictions_7e.csv).")
elif MLP_WAPE <= BEATS_LGBM_THRESHOLD:
    print(f"~  Gated MLP {MLP_WAPE:.3f} beats LGBM but is worse than 7a-MLP by "
          f"{MLP_WAPE - BEATS_MLP_THRESHOLD:+.2f}. L1 was probably too aggressive — "
          f"drop GATE_L1_LAMBDA from {GATE_L1_LAMBDA} to {GATE_L1_LAMBDA / 2} and re-run.")
else:
    print(f"✗  Gated MLP {MLP_WAPE:.3f} > LGBM. Unexpected. Most likely λ is way too high — "
          "check gate_mean above; if most gates are < 0.1 the L1 squashed even strong features. "
          f"Try GATE_L1_LAMBDA = 0.001 first.")

=== Learned covariate gates (sigmoid scalar per covariate) ===
  L1 λ = 0.01.  Averaged over 9 (fold × seed) runs.

  covariate                                   mean     std     verdict
  ----------------------------------------------------------------------
  queue_pressure_forecast                    0.233   0.013     PARTIAL
  event_load_forecast                        0.183   0.010        DROP
  network_pressure_forecast                  0.155   0.009        DROP
  maintenance_known                          0.100   0.006        DROP
  demand_forecast                            0.087   0.007        DROP
  shock_risk                                 0.080   0.006        DROP
  promotion_intensity                        0.074   0.005        DROP
  upstream_quality_forecast                  0.067   0.007        DROP
  workload_intensity                         0.060   0.008        DROP
  service_irregularity_risk_forecast         0.033   0.006        DROP
  unit_reliability_forecast   

---
## §8. Save artifacts + decision summary

In [9]:
oof_path = OUTPUT_DIR / "oof_predictions_7e_mlp_gated.csv"
oof_mlp.to_csv(oof_path, index=False)
print(f"OOF predictions → {oof_path}")

gate_rows = []
for (k, s, gates) in gate_vals_per_run:
    for i, name in enumerate(DOMAIN_FEAT_COLS):
        gate_rows.append({"fold": k, "seed": s, "feature": name, "gate": float(gates[i])})
gate_df = pd.DataFrame(gate_rows)
gate_summary = (gate_df.groupby("feature")["gate"]
                .agg(["mean", "std", "min", "max"])
                .reset_index()
                .sort_values("mean", ascending=False))
gates_path = OUTPUT_DIR / "gates_7e_mlp_gated.csv"
gate_summary.to_csv(gates_path, index=False)
print(f"Gate values → {gates_path}")
print()
print(gate_summary.to_string(index=False))

print("\n=== Decision summary ===")
print(f"  LGBM-MAE (§4)         : {LGBM_WAPE:.3f}")
print(f"  7a-tabular MLP (anchor): {BEATS_MLP_THRESHOLD:.2f}")
print(f"  Gated MLP (§6)        : {MLP_WAPE:.3f}  ({MLP_WAPE - BEATS_MLP_THRESHOLD:+.3f} vs 7a-MLP)")
print(f"  Pearson r vs LGBM     : {r_pooled:.4f}")
print(f"  Gates KEEP  (>0.60)   : {sum(1 for v in gate_mean if v > 0.60)} / 11")
print(f"  Gates PARTIAL         : {sum(1 for v in gate_mean if 0.20 < v <= 0.60)} / 11")
print(f"  Gates DROP  (≤0.20)   : {sum(1 for v in gate_mean if v <= 0.20)} / 11")

OOF predictions → /kaggle/working/oof_predictions_7e_mlp_gated.csv
Gate values → /kaggle/working/gates_7e_mlp_gated.csv

                           feature     mean      std      min      max
           queue_pressure_forecast 0.232649 0.013678 0.215215 0.249395
               event_load_forecast 0.183021 0.010958 0.170059 0.196882
         network_pressure_forecast 0.154851 0.009509 0.142010 0.167211
                 maintenance_known 0.100048 0.006487 0.091838 0.109432
                   demand_forecast 0.086863 0.007085 0.078892 0.096964
                        shock_risk 0.079665 0.006444 0.070655 0.090206
               promotion_intensity 0.074099 0.005641 0.068580 0.081946
         upstream_quality_forecast 0.066556 0.006903 0.057502 0.076190
                workload_intensity 0.060500 0.008299 0.050398 0.070943
service_irregularity_risk_forecast 0.032939 0.005844 0.026330 0.040164
         unit_reliability_forecast 0.029923 0.006124 0.022946 0.037376

=== Decision summary ===
 

---
## §9. Next step depending on the §7 verdict

- **Verdict = ✅ (CV-WAPE ≤ 14.52)**: write the 7f submission notebook. Refit the gated MLP on full train (5 seeds), apply to `validation_input.csv`, write `validation_predictions_7e.csv`. This is the pure-DL submission.
- **Verdict = ~ (between 14.52 and 14.78)**: re-run §6 with `GATE_L1_LAMBDA / 2`. The L1 squeezed too hard. Iterate until WAPE matches the 7a-MLP within 0.05, then ship.
- **Verdict = ✗ (worse than LGBM)**: check whether most gates collapsed to ~0. If yes, λ way too high → try 0.001. If gates look reasonable but WAPE is bad, something else is wrong — diff this notebook against 7a-tabular.
- **Bonus interpretation**: compare the `vs. data-analysis verdicts` table from §7. If the model's KEEPs are exactly the STRONG/MODERATE features from the static-R² analysis, the network learned what we already knew from pre-experiment exploration. If it disagrees (e.g., DROPs a STRONG feature, or KEEPs a WEAK one), that's a more interesting finding — the model is identifying signal structure the pooled-R² analysis missed.